# RAILGUN+ — Full Evaluation (Kaggle)

Four-method comparison on the **LaCAM-distilled** model:
- **expert** (LaCAM oracle) — upper reference (near-optimal)
- **pibt_only** (pure PIBT, no net) — THE BASELINE TO BEAT
- **greedy** (net, naive resolution) — baseline RAILGUN (deadlocks)
- **corrected** (net + PIBT corrector) — **ours**

Produces: CSR, deadlock-rate, SoC-ratio, and makespan tables + plots.
The key result is whether **corrected beats pibt_only** — which a LaCAM teacher makes possible.

## 1. Setup

In [ ]:
import os, sys, pickle
!git clone -q https://github.com/tay805/railgun-plus.git /kaggle/working/railgun-plus
!pip install -q pogema
sys.path.insert(0,'/kaggle/working/railgun-plus/src')
import torch

# EDIT: path to your trained checkpoint (from the training notebook's dataset)
CKPT='/kaggle/input/your-checkpoints/best.pt'
RESULTS_DIR='/kaggle/working/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

## 2. Load model (best.pt)

In [ ]:
from railgun_plus.models import RailgunUNet
device='cuda' if torch.cuda.is_available() else 'cpu'
model=RailgunUNet(6,5,base=64).to(device)
ck=torch.load(CKPT,map_location=device)
model.load_state_dict(ck['model']);model.eval()
print('loaded epoch',ck.get('epoch'),'on',device)

## 3. Build bigger, fixed test sets (resumable, saved to disk)
50 instances/agent-count so high-agent numbers aren't noisy. Note: test instances are solved by PIBT just to confirm solvability + get a horizon; the EXPERT method re-solves them.

In [ ]:
from railgun_plus.data.generate import generate_with_pogema
AGENT_COUNTS=[16,32,64,96,128]
N_PER=50
TEST_PATH=f'{RESULTS_DIR}/test_sets.pkl'
test_sets={}
if os.path.exists(TEST_PATH):
    test_sets=pickle.load(open(TEST_PATH,'rb'))
for k in AGENT_COUNTS:
    if test_sets.get(k):
        print(k,'agents: have',len(test_sets[k]),'(skip)'); continue
    test_sets[k]=generate_with_pogema(N_PER,32,0.2,k,seed=1000+k)
    print(k,'agents:',len(test_sets[k]),'generated')
    pickle.dump(test_sets,open(TEST_PATH,'wb'))
print('test sets:',{k:len(v) for k,v in test_sets.items()})

## 4. Run the full sweep (all four methods)

In [ ]:
from railgun_plus.eval.harness import run_sweep, sweep_to_table, plot_sweep
sweep=run_sweep(model,test_sets,device=device)

## 5. Results table (CSR / deadlock-rate / SoC-ratio / makespan)

In [ ]:
import pandas as pd
rows=sweep_to_table(sweep,out_csv=f'{RESULTS_DIR}/results_table.csv')
df=pd.DataFrame(rows)
print('=== CSR (higher=better) ==='); display(df.pivot(index='agents',columns='method',values='csr'))
print('=== Deadlock rate (lower=better) ==='); display(df.pivot(index='agents',columns='method',values='deadlock_rate'))
print('=== SoC ratio (closer to 1=better; solved only) ==='); display(df.pivot(index='agents',columns='method',values='avg_soc_ratio'))
df

## 6. Headline plot — CSR vs agents
**The key comparison: does `corrected` (ours) beat `pibt_only` (free baseline)?**

In [ ]:
plot_sweep(sweep,metric='csr',title='CSR vs agents (higher=better)',savepath=f'{RESULTS_DIR}/csr.png')

## 7. Deadlock efficiency — deadlock rate vs agents
This is the headline novelty: greedy RAILGUN deadlocks badly; the corrector stays low.

In [ ]:
plot_sweep(sweep,metric='deadlock_rate',title='Deadlock rate vs agents (lower=better)',savepath=f'{RESULTS_DIR}/deadlock.png')

## 8. Solution quality — SoC / lower-bound

In [ ]:
plot_sweep(sweep,metric='avg_soc_ratio_solved',title='SoC ratio (1.0=optimal; solved only)',savepath=f'{RESULTS_DIR}/soc_ratio.png')

## 9. Makespan (paper Table I style)

In [ ]:
plot_sweep(sweep,metric='avg_makespan_solved',title='Makespan vs agents (solved)',savepath=f'{RESULTS_DIR}/makespan.png')

## 10. (Later) POGEMA-harness comparison vs SCRIMP/DCC/MAPF-GPT
Run your model through pogema-toolbox's official benchmark to land on the same radar as published baselines.

In [ ]:
# from railgun_plus.eval.harness import pogema_benchmark_stub  # roadmap in docstring